# Stelow cohort — apc panel (Panel 4)

**Goal:** Characterize immune cell infiltration in MHC II–high vs MHC II–low
LUAD tumors using multiplexed immunofluorescence (Panel 4: DAPI, CD20, CD68,
CD208, cKIT, MHC II, PanCK) from large tissue sections. Cell densities are
normalized by tissue area (cells/mm²) per region (tumor, stroma, alveoli)
and compared by Mann-Whitney U test with Benjamini-Hochberg FDR correction.

**Input:** Panel 3 HALO summary CSV (one row per ROI), Stelow cohort metadata CSV.

**Output:** Figure 5C (PanCK+MHCII+/- stacked bar), Figure 5D (APC
density by region); supplementary tables.

In [ ]:
from pathlib import Path
import yaml
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

fig_out   = fig_dir   / 'figure5'
supp_out  = fig_dir   / 'figure5-supp'
table_out = table_dir / 'figure5'

fig_out.mkdir(parents=True, exist_ok=True)
supp_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

In [ ]:
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']

custom_palette = {
    'class II high': cmap_high_low[0],
    'class II low':  cmap_high_low[1],
}

## Load data

Panel 4 HALO summary CSV contains one row per ROI with cell counts and tissue
area per classifier region (tumor, stroma, alveoli). Patient and ROI identifiers
are extracted from the image location string. Stelow cohort clinical metadata
is merged on patient code.

In [ ]:
panel4_path  = Path('/project/Dolatshahi_Lab/Gabe_Hanson/data/mhc2_vectra_data/object_data/stelow_slides/full_image_set/panel4/summary_data_pan4.csv')
metadata_path = Path('/project/Dolatshahi_Lab/Gabe_Hanson/stelow_moore/vectra_stelow_slide_metadata.csv')

panel4   = pd.read_csv(panel4_path)
metadata = pd.read_csv(metadata_path)

# extract ROI and patient identifiers from image location string
panel4['ROI']       = panel4['Image Location'].str.extract(r'(S\d{1,2} LUNG.*?\[.*?\])')
panel4['PatientID'] = panel4['Image Location'].str.extract(r'(S\d{1,2} LUNG)')

print(f"Panel 4 ROIs loaded: {len(panel4):,}")
print(f"Patients: {panel4['PatientID'].nunique()}")

## Merge metadata and filter to LUAD

Clinical metadata is merged on patient code. Analysis is restricted to
adenocarcinoma (HIST == 'Adenocarcinoma'). Key metadata columns retained
for downstream use include histology, stage, sex, age, smoking history,
PD-L1 score, and MHC I pathology score (Stelow MHC).

In [ ]:
# match PatientID format to metadata Code column
panel4['Code'] = panel4['PatientID'].str.replace(' LUNG', '', regex=False)

metadata_cols = [
    'Code', 'HIST', 'SEX', 'AGE', 'STAGE', 'SMOKE_HX', 'SMOKE_PY',
    'PD-L1 Score', 'Stelow MHC', 'RACE', 'ETHN'
]
combined = panel4.merge(metadata[metadata_cols], on='Code', how='left')

# filter to LUAD
luad = combined.loc[combined['HIST'] == 'Adenocarcinoma'].copy()

print(f"LUAD ROIs: {len(luad):,}")
print(f"LUAD patients: {luad['PatientID'].nunique()}")
print(luad['PatientID'].unique())

In [ ]:
# fix HALO export typo — C20 → CD20
luad = luad.rename(columns={
    col: col.replace('C20+MHCII-', 'CD20+MHCII-')
    for col in luad.columns if 'C20+MHCII-' in col
})

In [ ]:
# derive total cell type columns by summing MHC II+ and MHC II- subsets
for region in ['tumor', 'stroma', 'alveoli']:
    # total macrophages (CD68+)
    luad[f'{region}: Total CD68+ Cells'] = (
        luad[f'{region}: PanCK-MHCII+CD68+CKIT- Cells'] +
        luad[f'{region}: MHCII-PanCK-CD68+CKIT- Cells']
    )
    # total CKit+ cells
    luad[f'{region}: Total CKIT+ Cells'] = (
        luad[f'{region}: PanCK-MHCII+CD68-CKIT+ Cells'] +
        luad[f'{region}: MHCII-PanCK-CD68-CKIT+ Cells']
    )
    # total B cells (CD20+)
    luad[f'{region}: Total CD20+ Cells'] = (
        luad[f'{region}: CD20+MHCII+PanCK-CD68-CKIT- Cells'] +
        luad[f'{region}: CD20+MHCII-PanCK-CD68-CKIT- Cells']
    )
    # total CD68+CKIT+ double positive
    luad[f'{region}: Total CD68+CKIT+ Cells'] = (
        luad[f'{region}: PanCK-MHCII+CD68+CKIT+ Cells'] +
        luad[f'{region}: MHCII-PanCK-CD68+CKIT+ Cells']
    )
    # total CD208+ (mature DCs)
    luad[f'{region}: Total CD208+ Cells'] = (
        luad[f'{region}: CD208+MHCII+ Cells']
        # CD208+MHCII- not in panel — CD208 is essentially always MHCII+
    )

In [ ]:
# compute per-patient PanCK+MHCII+ fraction
patient_mhc2 = (
    luad.groupby('PatientID')[['tumor: PanCK+MHCII+ Cells', 'tumor: PanCK+MHCII- Cells']]
    .sum()
    .reset_index()
)
patient_mhc2['mhc2_pos_fraction'] = (
    patient_mhc2['tumor: PanCK+MHCII+ Cells'] /
    (patient_mhc2['tumor: PanCK+MHCII+ Cells'] + patient_mhc2['tumor: PanCK+MHCII- Cells'])
)
patient_mhc2 = patient_mhc2.sort_values('mhc2_pos_fraction', ascending=False)
print(patient_mhc2[['PatientID', 'mhc2_pos_fraction']].to_string(index=False))

# plot distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    range(len(patient_mhc2)),
    patient_mhc2['mhc2_pos_fraction'],
    color=[cmap_high_low[0] if f > patient_mhc2['mhc2_pos_fraction'].mean() 
           else cmap_high_low[1] for f in patient_mhc2['mhc2_pos_fraction']]
)
ax.axhline(patient_mhc2['mhc2_pos_fraction'].mean(), 
           color='gray', linestyle='--', lw=1.5, label='mean')
ax.set_xticks(range(len(patient_mhc2)))
ax.set_xticklabels(patient_mhc2['PatientID'].str.replace(' LUNG', ''), rotation=45, ha='right')
ax.set_ylabel('PanCK+MHCII+ fraction')
ax.legend(frameon=False)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

## MHC II patient classification

Patients are classified as MHC II high or low based on the fraction of
PanCK+ tumor cells expressing MHC II. A threshold of 0.4 is used, which
corresponds to a natural break in the distribution between S21 (0.57) and
S18 (0.26). This threshold is applied to the per-patient aggregated MHC II+
fraction across all tumor ROIs.

In [ ]:
MHCII_THRESHOLD = 0.4
EXCLUDE_PATIENTS = ['S21 LUNG']

# compute per-patient MHC II+ fraction from aggregated tumor counts
patient_mhc2 = (
    luad.groupby('PatientID')[['tumor: PanCK+MHCII+ Cells', 'tumor: PanCK+MHCII- Cells']]
    .sum()
    .reset_index()
)
patient_mhc2['mhc2_pos_fraction'] = (
    patient_mhc2['tumor: PanCK+MHCII+ Cells'] /
    (patient_mhc2['tumor: PanCK+MHCII+ Cells'] + patient_mhc2['tumor: PanCK+MHCII- Cells'])
)

# exclude borderline case
patient_mhc2 = patient_mhc2[~patient_mhc2['PatientID'].isin(EXCLUDE_PATIENTS)].copy()
luad = luad[~luad['PatientID'].isin(EXCLUDE_PATIENTS)].copy()

patient_mhc2['patient classification'] = np.where(
    patient_mhc2['mhc2_pos_fraction'] >= MHCII_THRESHOLD,
    'class II high', 'class II low'
)

# merge classification back to luad
luad = luad.merge(
    patient_mhc2[['PatientID', 'patient classification']],
    on='PatientID', how='left'
)

print(luad['patient classification'].value_counts())
print("\nClassification per patient:")
print(
    patient_mhc2[['PatientID', 'mhc2_pos_fraction', 'patient classification']]
    .sort_values('mhc2_pos_fraction', ascending=False)
    .to_string(index=False)
)
patient_mhc2.to_csv(table_out / 'stelow_panel4_patient_mhc2_classification.csv', index=False)

In [ ]:
hardcoded = {
    'class II high': ['S1 LUNG','S3 LUNG', 'S5 LUNG', 'S8 LUNG', 'S16 LUNG','S21 LUNG'],
    'class II low':  ['S2 LUNG', 'S4 LUNG', 'S7 LUNG', 'S10 LUNG','S13 LUNG', 'S17 LUNG', 'S18 LUNG', 'S19 LUNG','S22 LUNG']
}

for patient, cls in {p: c for c, ps in hardcoded.items() for p in ps}.items():
    our_cls = patient_mhc2.loc[patient_mhc2['PatientID'] == patient, 'patient classification'].values
    if len(our_cls) > 0 and our_cls[0] != cls:
        print(f"MISMATCH: {patient} — hardcoded: {cls}, ours: {our_cls[0]}")
    
print("Check complete")

## Note on S21 exclusion

S21 LUNG showed discordant MHC II classification between panel 3
(PanCK+MHCII+ fraction = 0.218, class II low) and panel 4
(PanCK+MHCII+ fraction = 0.570, class II high) despite being adjacent
tissue sections from the same tumor block. Within-patient ROI variance
for S21 was low (CV = 0.40), ruling out intratumoral MHC II heterogeneity
as the explanation. The discordance is attributed to staining variability
between panels for a borderline case near the classification threshold.
S21 is excluded from all comparative analyses in both panel 3 and panel 4
notebooks. Both versions (with and without S21) are available as separate
notebooks for sensitivity analysis.

In [ ]:
# --- Figure 6 cell types — APC panel by region ---

# primary APC cell types — MHC II+ APCs (the biological question)
cell_cols_fig6_tumor = [
    'tumor: PanCK-MHCII+CD68+CKIT- Cells',      # macrophage MHCII+
    'tumor: MHCII-PanCK-CD68+CKIT- Cells',       # macrophage MHCII-
    'tumor: Total CD68+ Cells',                   # macrophage total
    'tumor: CD20+MHCII+PanCK-CD68-CKIT- Cells',  # B cell MHCII+
    'tumor: CD20+MHCII-PanCK-CD68-CKIT- Cells',  # B cell MHCII-
    'tumor: Total CD20+ Cells',                   # B cell total
    'tumor: CD208+MHCII+ Cells',                  # mature DC
    'tumor: PanCK-MHCII+CD68-CKIT+ Cells',       # CKit+ MHCII+
    'tumor: MHCII-PanCK-CD68-CKIT+ Cells',       # CKit+ MHCII-
    'tumor: Total CKIT+ Cells',                   # CKit+ total
    'tumor: PanCK-MHCII+CD68+CKIT+ Cells',       # CD68+CKit+ MHCII+
    'tumor: MHCII-PanCK-CD68+CKIT+ Cells',       # CD68+CKit+ MHCII-
    'tumor: Total CD68+CKIT+ Cells',              # CD68+CKit+ total
]

cell_cols_fig6_stroma = [
    'stroma: PanCK-MHCII+CD68+CKIT- Cells',      
    'stroma: MHCII-PanCK-CD68+CKIT- Cells',      
    'stroma: Total CD68+ Cells',                 
    'stroma: CD20+MHCII+PanCK-CD68-CKIT- Cells', 
    'stroma: CD20+MHCII-PanCK-CD68-CKIT- Cells', 
    'stroma: Total CD20+ Cells',                 
    'stroma: CD208+MHCII+ Cells',                
    'stroma: PanCK-MHCII+CD68-CKIT+ Cells',      
    'stroma: MHCII-PanCK-CD68-CKIT+ Cells',      
    'stroma: Total CKIT+ Cells',                 
    'stroma: PanCK-MHCII+CD68+CKIT+ Cells',      
    'stroma: MHCII-PanCK-CD68+CKIT+ Cells',      
    'stroma: Total CD68+CKIT+ Cells',            
]

cell_cols_fig6_alveoli = [
    'alveoli: PanCK-MHCII+CD68+CKIT- Cells',      
    'alveoli: MHCII-PanCK-CD68+CKIT- Cells',      
    'alveoli: Total CD68+ Cells',                 
    'alveoli: CD20+MHCII+PanCK-CD68-CKIT- Cells', 
    'alveoli: CD20+MHCII-PanCK-CD68-CKIT- Cells', 
    'alveoli: Total CD20+ Cells',                 
    'alveoli: CD208+MHCII+ Cells',                
    'alveoli: PanCK-MHCII+CD68-CKIT+ Cells',      
    'alveoli: MHCII-PanCK-CD68-CKIT+ Cells',      
    'alveoli: Total CKIT+ Cells',                 
    'alveoli: PanCK-MHCII+CD68+CKIT+ Cells',      
    'alveoli: MHCII-PanCK-CD68+CKIT+ Cells',      
    'alveoli: Total CD68+CKIT+ Cells',            
]

# supplementary — MHC validation + MHCII- APCs
cell_cols_supp_tumor = [
    'tumor: PanCK+MHCII+ Cells',
    'tumor: PanCK+MHCII- Cells',
    'tumor: MHCII-PanCK-CD68+CKIT+ Cells',
]

cell_cols_supp_stroma = [
    'stroma: PanCK+MHCII+ Cells',
    'stroma: PanCK+MHCII- Cells',
    'stroma: MHCII-PanCK-CD68+CKIT+ Cells',
]

cell_cols_supp_alveoli = [
    'alveoli: PanCK+MHCII+ Cells',
    'alveoli: PanCK+MHCII- Cells',
    'alveoli: MHCII-PanCK-CD68+CKIT+ Cells',
]

area_cols = ['tumor Area (mm²)', 'stroma Area (mm²)', 'alveoli Area (mm²)']

all_cell_cols = (
    cell_cols_fig6_tumor + cell_cols_fig6_stroma + cell_cols_fig6_alveoli +
    cell_cols_supp_tumor + cell_cols_supp_stroma + cell_cols_supp_alveoli
)

## Figure 6C — APC panel cell type density by region

MHC II+ and MHC II– APC populations (macrophages, B cells, mature DCs,
CKit+ cells) compared between MHC II–high and MHC II–low patients across
tumor, stroma, and alveoli regions. Cell densities normalized by tissue
area (cells/mm²). FDR-adjusted p-values from Benjamini-Hochberg correction
applied separately per region.

In [ ]:
luad_classified = luad.loc[
    luad['patient classification'].isin(['class II high', 'class II low'])
].copy()

# normalize each ROI individually by region area
normalized = luad_classified.copy()
for col in all_cell_cols:
    if col.startswith('tumor'):
        normalized[col] = luad_classified[col].values / luad_classified['tumor Area (mm²)'].values
    elif col.startswith('stroma'):
        normalized[col] = luad_classified[col].values / luad_classified['stroma Area (mm²)'].values
    elif col.startswith('alveoli'):
        normalized[col] = luad_classified[col].values / luad_classified['alveoli Area (mm²)'].values

normalized = normalized.replace([np.inf, -np.inf], np.nan)

print(f"ROIs in analysis: {len(normalized):,}")
print(normalized['patient classification'].value_counts())

In [ ]:
from ceiba.stats_utils import run_stats

stats_fig6_tumor,   pval_map_fig6_tumor   = run_stats(normalized, cell_cols_fig6_tumor)
stats_fig6_stroma,  pval_map_fig6_stroma  = run_stats(normalized, cell_cols_fig6_stroma)
stats_fig6_alveoli, pval_map_fig6_alveoli = run_stats(normalized, cell_cols_fig6_alveoli)

stats_supp_tumor,   pval_map_supp_tumor   = run_stats(normalized, cell_cols_supp_tumor)
stats_supp_stroma,  pval_map_supp_stroma  = run_stats(normalized, cell_cols_supp_stroma)
stats_supp_alveoli, pval_map_supp_alveoli = run_stats(normalized, cell_cols_supp_alveoli)

stats_fig6_tumor.to_csv(table_out   / 'stelow_panel4_apc_tumor_stats.csv', index=False)
stats_fig6_stroma.to_csv(table_out  / 'stelow_panel4_apc_stroma_stats.csv', index=False)
stats_fig6_alveoli.to_csv(table_out / 'stelow_panel4_apc_alveoli_stats.csv', index=False)

print("=== Figure 6C — APC tumor ===")
print(stats_fig6_tumor[['Cell Type', 'P-value', 'FDR-adjusted P-value', 'Significant (FDR < 0.05)']].to_string(index=False))

In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from ceiba.plot_utils import make_comparison_figure

# prepare plot_df
plot_df = normalized.copy()
for col in all_cell_cols:
    plot_df[f'{col}_fraction'] = plot_df[col]

# tumor
fig, axes = make_comparison_figure(
    plot_df,
    cell_types=cell_cols_fig6_tumor,
    neg_label='class II low',
    pos_label='class II high',
    neg_color=cmap_high_low[1],
    pos_color=cmap_high_low[0],
    ncols=len(cell_cols_fig6_tumor),
    panel_size=(3.0, 3.0),
    pval_map=pval_map_fig6_tumor,
    ylabel='Density (cells/mm²)',
    point_alpha=0.1,
    title_fontsize=9,
    label_fontsize=10,
    tick_fontsize=9,
    pval_fontsize=8,
)
plt.show()

# stroma
fig, axes = make_comparison_figure(
    plot_df,
    cell_types=cell_cols_fig6_stroma,
    neg_label='class II low',
    pos_label='class II high',
    neg_color=cmap_high_low[1],
    pos_color=cmap_high_low[0],
    ncols=len(cell_cols_fig6_stroma),
    panel_size=(3.0, 3.0),
    pval_map=pval_map_fig6_stroma,
    ylabel='Density (cells/mm²)',
    point_alpha=0.1,
    title_fontsize=9,
    label_fontsize=10,
    tick_fontsize=9,
    pval_fontsize=8,
)
plt.show()

# alveoli
fig, axes = make_comparison_figure(
    plot_df,
    cell_types=cell_cols_fig6_alveoli,
    neg_label='class II low',
    pos_label='class II high',
    neg_color=cmap_high_low[1],
    pos_color=cmap_high_low[0],
    ncols=len(cell_cols_fig6_alveoli),
    panel_size=(3.0, 3.0),
    pval_map=pval_map_fig6_alveoli,
    ylabel='Density (cells/mm²)',
    point_alpha=0.1,
    title_fontsize=9,
    label_fontsize=10,
    tick_fontsize=9,
    pval_fontsize=8,
)
plt.show()

In [ ]:
print("=== Tumor ===")
print(stats_fig6_tumor[['Cell Type', 'FDR-adjusted P-value', 'Higher group', 'Significant (FDR < 0.05)']].to_string(index=False))
print("\n=== Stroma ===")
print(stats_fig6_stroma[['Cell Type', 'FDR-adjusted P-value', 'Higher group', 'Significant (FDR < 0.05)']].to_string(index=False))
print("\n=== Alveoli ===")
print(stats_fig6_alveoli[['Cell Type', 'FDR-adjusted P-value', 'Higher group', 'Significant (FDR < 0.05)']].to_string(index=False))

## Figure 6C — APC panel primary figure

The full APC panel analysis above served as an exploratory overview of all
cell type comparisons across regions. For the primary figure we focus on
the most biologically interpretable comparisons — macrophage MHC II+/–
subsets and totals in tumor, and mature DC and CKit+ enrichment in stroma,
where the signals are strongest and most consistent. Full results for all
cell types and regions are shown in the supplementary figure below.

In [ ]:
# primary figure cell type lists — most interpretable comparisons
cell_cols_fig6_primary_tumor = [
    'tumor: PanCK-MHCII+CD68+CKIT- Cells',    # macrophage MHCII+
    'tumor: MHCII-PanCK-CD68+CKIT- Cells',    # macrophage MHCII-
    'tumor: Total CD68+ Cells',                # macrophage total
    'tumor: Total CD20+ Cells',                # B cell total
]

cell_cols_fig6_primary_stroma = [
    'stroma: PanCK-MHCII+CD68+CKIT- Cells',   # macrophage MHCII+
    'stroma: MHCII-PanCK-CD68+CKIT- Cells',   # macrophage MHCII-
    'stroma: Total CD68+ Cells',               # macrophage total
    'stroma: CD208+MHCII+ Cells',              # mature DC
    'stroma: Total CKIT+ Cells',               # CKit+ total
]

# run stats for primary
stats_primary_tumor,  pval_map_primary_tumor  = run_stats(normalized, cell_cols_fig6_primary_tumor)
stats_primary_stroma, pval_map_primary_stroma = run_stats(normalized, cell_cols_fig6_primary_stroma)

# prepare plot_df
plot_df = normalized.copy()
for col in cell_cols_fig6_primary_tumor + cell_cols_fig6_primary_stroma:
    plot_df[f'{col}_fraction'] = plot_df[col]

# tumor
fig, axes = make_comparison_figure(
    plot_df,
    cell_types=cell_cols_fig6_primary_tumor,
    neg_label='class II low',
    pos_label='class II high',
    neg_color=cmap_high_low[1],
    pos_color=cmap_high_low[0],
    ncols=len(cell_cols_fig6_primary_tumor),
    panel_size=(3.0, 3.0),
    pval_map=pval_map_primary_tumor,
    ylabel='Density (cells/mm²)',
    point_alpha=0.1,
    title_fontsize=9,
    label_fontsize=10,
    tick_fontsize=9,
    pval_fontsize=8,
    outpath=fig_out / 'fig6c_primary_tumor.pdf',
)
plt.show()

# stroma
fig, axes = make_comparison_figure(
    plot_df,
    cell_types=cell_cols_fig6_primary_stroma,
    neg_label='class II low',
    pos_label='class II high',
    neg_color=cmap_high_low[1],
    pos_color=cmap_high_low[0],
    ncols=len(cell_cols_fig6_primary_stroma),
    panel_size=(3.0, 3.0),
    pval_map=pval_map_primary_stroma,
    ylabel='Density (cells/mm²)',
    point_alpha=0.1,
    title_fontsize=9,
    label_fontsize=10,
    tick_fontsize=9,
    pval_fontsize=8,
    outpath=fig_out / 'fig6c_primary_stroma.pdf',
)
plt.show()

## Supplementary — full APC panel all regions

Complete cell type density comparisons for all APC phenotypes across
tumor, stroma, and alveoli regions. FDR-adjusted p-values from
Benjamini-Hochberg correction applied separately per region.

In [ ]:
# full supplementary — everything including MHC validation
cell_cols_supp_fig6_tumor = (
    cell_cols_fig6_tumor +
    cell_cols_supp_tumor
)

cell_cols_supp_fig6_stroma = (
    cell_cols_fig6_stroma +
    cell_cols_supp_stroma
)

cell_cols_supp_fig6_alveoli = (
    cell_cols_fig6_alveoli +
    cell_cols_supp_alveoli
)

# run stats for full supp sets
stats_supp_full_tumor,   pval_map_supp_full_tumor   = run_stats(normalized, cell_cols_supp_fig6_tumor)
stats_supp_full_stroma,  pval_map_supp_full_stroma  = run_stats(normalized, cell_cols_supp_fig6_stroma)
stats_supp_full_alveoli, pval_map_supp_full_alveoli = run_stats(normalized, cell_cols_supp_fig6_alveoli)

# prepare plot_df
for col in cell_cols_supp_fig6_tumor + cell_cols_supp_fig6_stroma + cell_cols_supp_fig6_alveoli:
    if f'{col}_fraction' not in plot_df.columns:
        plot_df[f'{col}_fraction'] = plot_df[col]

# tumor
fig, axes = make_comparison_figure(
    plot_df,
    cell_types=cell_cols_supp_fig6_tumor,
    neg_label='class II low',
    pos_label='class II high',
    neg_color=cmap_high_low[1],
    pos_color=cmap_high_low[0],
    ncols=len(cell_cols_supp_fig6_tumor),
    panel_size=(3.0, 3.0),
    pval_map=pval_map_supp_full_tumor,
    ylabel='Density (cells/mm²)',
    point_alpha=0.1,
    title_fontsize=9,
    label_fontsize=10,
    tick_fontsize=9,
    pval_fontsize=8,
    outpath=supp_out / 'figS6_apc_full_tumor.pdf',
)
plt.show()

# stroma
fig, axes = make_comparison_figure(
    plot_df,
    cell_types=cell_cols_supp_fig6_stroma,
    neg_label='class II low',
    pos_label='class II high',
    neg_color=cmap_high_low[1],
    pos_color=cmap_high_low[0],
    ncols=len(cell_cols_supp_fig6_stroma),
    panel_size=(3.0, 3.0),
    pval_map=pval_map_supp_full_stroma,
    ylabel='Density (cells/mm²)',
    point_alpha=0.1,
    title_fontsize=9,
    label_fontsize=10,
    tick_fontsize=9,
    pval_fontsize=8,
    outpath=supp_out / 'figS6_apc_full_stroma.pdf',
)
plt.show()

# alveoli
fig, axes = make_comparison_figure(
    plot_df,
    cell_types=cell_cols_supp_fig6_alveoli,
    neg_label='class II low',
    pos_label='class II high',
    neg_color=cmap_high_low[1],
    pos_color=cmap_high_low[0],
    ncols=len(cell_cols_supp_fig6_alveoli),
    panel_size=(3.0, 3.0),
    pval_map=pval_map_supp_full_alveoli,
    ylabel='Density (cells/mm²)',
    point_alpha=0.1,
    title_fontsize=9,
    label_fontsize=10,
    tick_fontsize=9,
    pval_fontsize=8,
    outpath=supp_out / 'figS6_apc_full_alveoli.pdf',
)
plt.show()

## Mixed effects model — APC panel

Mann-Whitney U tests on ROI-level data show highly significant differences
for many APC cell types, but the large number of ROIs per patient means
these tests are heavily influenced by within-patient variance rather than
true between-patient differences. To account for the non-independence of
ROIs within patients, we fit linear mixed effects models with patient as
a random intercept. Log10-transformed cell density is modeled as a function
of MHC II classification. This approach provides effect size estimates
(coefficients) with 95% confidence intervals that correctly reflect the
between-patient variance, giving a more conservative and statistically
principled assessment of the classification effect. Given n=13 patients
(5 high, 8 low), the mixed effects model is underpowered to detect all but
the largest effects — results should be interpreted as hypothesis-generating
with effect direction and magnitude being more informative than p-values.
Results are visualized as a forest plot showing log10 fold change in cell
density (class II high vs low) with 95% CI. Filled points indicate
FDR-adjusted p < 0.05; open points are non-significant.

In [ ]:
from ceiba.stats_utils import run_stats, run_mixed_effects


# run mixed effects for APC columns
fig6_cols = (
    cell_cols_fig6_primary_tumor +
    cell_cols_fig6_primary_stroma
)

me_results_fig6 = run_mixed_effects(normalized, fig6_cols)
me_results_fig6.to_csv(table_out / 'stelow_panel4_apc_mixed_effects.csv', index=False)
print(me_results_fig6[['Cell Type', 'Coefficient', 'P-value', 'FDR-adjusted P-value', 'Significant (FDR < 0.05)']].to_string(index=False))

In [ ]:
from ceiba.plot_utils import plot_forest

fig = plot_forest(
    me_results_fig6,
    fig_path=supp_out / 'figS6_stelow_apc_mixed_effects_forest.pdf',
    title='Mixed effects model — APC density by region',
    region_colors={
        'tumor':   cmap_high_low[2],
        'stroma':  cmap_high_low[3],
        'alveoli': cmap_high_low[4],
    },
    legend_anchor=(1.2, 0.02),

)

In [ ]:
# run mixed effects for all APC columns
fig6_all_cols = (
    cell_cols_fig6_tumor +
    cell_cols_fig6_stroma +
    cell_cols_fig6_alveoli
)

me_results_fig6_all = run_mixed_effects(normalized, fig6_all_cols)
me_results_fig6_all.to_csv(table_out / 'stelow_panel4_apc_mixed_effects_all.csv', index=False)

print(me_results_fig6_all[['Cell Type', 'Coefficient', 'P-value', 'FDR-adjusted P-value', 'Significant (FDR < 0.05)']].to_string(index=False))

# forest plot — all cell types
fig = plot_forest(
    me_results_fig6_all,
    fig_path=supp_out / 'figS6_stelow_apc_mixed_effects_forest_all.pdf',
    title='Mixed effects model — APC density by region',
    row_height=0.45,
    region_colors={
        'tumor':   cmap_high_low[2],
        'stroma':  cmap_high_low[3],
        'alveoli': cmap_high_low[4],
    },
)
plt.show()

# note convergence warnings here - expected with only 13 patients

## Discussion notes — Stelow cohort MIF APC panel (Panel 4)

**Macrophage findings:**
- MHC II+ macrophages (PanCK-MHCII+CD68+CKIT-) are enriched in MHC II high
  tumors and stroma — consistent with a broadly activated antigen presentation
  environment where tumor MHC II expression co-occurs with macrophage MHC II
  upregulation
- MHC II- macrophages (MHCII-PanCK-CD68+CKIT-) are enriched in MHC II low
  tumors — suggesting impaired macrophage activation rather than macrophage
  exclusion in MHC II low tumors
- Total macrophage burden (Total CD68+) is higher in MHC II high in stroma
  but the direction is less clear in tumor — macrophage polarization state
  may matter more than total numbers
- The MHC II+/- macrophage split mirrors the tumor MHC II phenotype,
  suggesting a coordinated immune activation state rather than independent
  regulation of tumor and macrophage MHC II

**Mature DC findings:**
- CD208+ (DC-LAMP) mature DCs are significantly enriched in stroma of
  MHC II high tumors but not in tumor parenchyma — consistent with DC
  exclusion from the tumor compartment even in immunologically active tumors
- DC enrichment in stroma of MHC II high tumors suggests active antigen
  cross-presentation in the peritumoral space, which may drive the T cell
  infiltration seen in panel 3

**B cell findings:**
- Total B cells (Total CD20+) are enriched in MHC II low tumors, driven
  primarily by MHC II- B cells — may reflect dysfunctional or exhausted
  B cell compartment in immune cold tumors
- CD20+MHCII+ B cells are not significantly different between groups in
  tumor — the B cell MHC II signal is weaker than macrophage and DC signals

**CKit+ cell findings:**
- Total CKit+ cells enriched in MHC II high stroma — CKit+ cells in this
  context likely represent mast cells or DC progenitors, consistent with
  active immune recruitment in MHC II high tumors
- The CKit+CD68+ double positive population enriched in MHC II high suggests
  a myeloid progenitor or alternatively activated myeloid subset that
  warrants further characterization

**Mixed effects model:**
- Mixed effects model (n=13 patients) shows consistent directionality for
  macrophage and DC comparisons but wide confidence intervals — all effects
  cross zero, confirming the study is underpowered at patient level
- Some models showed convergence warnings due to small n — results are
  directional estimates only and should be interpreted alongside the
  ROI-level Mann-Whitney results
- The coordinated enrichment of MHC II+ macrophages, mature DCs, and CKit+
  cells in MHC II high tumors points toward a broader immune activation
  phenotype rather than isolated tumor MHC II upregulation

**Overall APC panel conclusion:**
- MHC II high tumors show a coordinated enrichment of professional APCs
  (MHC II+ macrophages, mature DCs) in stroma, consistent with active
  antigen presentation in the tumor microenvironment
- MHC II low tumors have more MHC II- macrophages and B cells — suggesting
  an immunosuppressed rather than immune-excluded phenotype
- Together with panel 3 lymphocyte findings, the data support a model where
  MHC II high tumors have both more professional APCs and more lymphocyte
  infiltration, while MHC II low tumors have dysfunctional APCs and reduced
  lymphocyte recruitment